- This notebook uses the raw player data to construct a knowledge graph without the temporal meta data.

In [188]:
import pandas as pd
import numpy as np
import re
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', None)

# libraries for creating name spaces
import rdflib
from urllib.parse import quote
from rdflib import Graph, URIRef, Literal, BNode, Namespace
from rdflib.namespace import FOAF, RDF, OWL, RDFS, XSD

In [189]:
raw_data = pd.read_csv('./Data/playerRawData.csv')
raw_data.head()

,player,fullName,team,teamCountry,DOB,sex,transferMarketID,startTime,endTime
0,http://www.wikidata.org/entity/Q730284,Gökhan Gönül,http://www.wikidata.org/entity/Q28380902,http://www.wikidata.org/entity/Q43,1985-01-04T00:00:00Z,http://www.wikidata.org/entity/Q6581097,33793,2016-01-01T00:00:00Z,2020-01-01T00:00:00Z
1,http://www.wikidata.org/entity/Q730284,Gökhan Gönül,http://www.wikidata.org/entity/Q28380902,http://www.wikidata.org/entity/Q43,1985-01-07T00:00:00Z,http://www.wikidata.org/entity/Q6581097,33793,2016-01-01T00:00:00Z,2020-01-01T00:00:00Z
2,http://www.wikidata.org/entity/Q730284,Gökhan Gönül,http://www.wikidata.org/entity/Q641373,http://www.wikidata.org/entity/Q43,1985-01-04T00:00:00Z,http://www.wikidata.org/entity/Q6581097,33793,2003-01-01T00:00:00Z,2004-01-01T00:00:00Z
3,http://www.wikidata.org/entity/Q730284,Gökhan Gönül,http://www.wikidata.org/entity/Q641373,http://www.wikidata.org/entity/Q43,1985-01-07T00:00:00Z,http://www.wikidata.org/entity/Q6581097,33793,2003-01-01T00:00:00Z,2004-01-01T00:00:00Z
4,http://www.wikidata.org/entity/Q730284,Gökhan Gönül,http://www.wikidata.org/entity/Q340133,http://www.wikidata.org/entity/Q43,1985-01-04T00:00:00Z,http://www.wikidata.org/entity/Q6581097,33793,2006-01-01T00:00:00Z,2006-01-01T00:00:00Z


In [190]:
len(raw_data.team.unique())

3031

In [191]:
raw_data.dtypes

player              object
fullName            object
team                object
teamCountry         object
DOB                 object
sex                 object
transferMarketID     int64
startTime           object
endTime             object
dtype: object

In [192]:
#invalid End times
mask_invalid_endtime = raw_data.endTime.str.len()>20
raw_data.loc[mask_invalid_endtime,'endTime']=" "

In [193]:
#adding namespace for tansferMarketID
raw_data['transferMarketID'] = raw_data['transferMarketID'].astype(str)
raw_data.loc[:,'transferMarketID'] = 'https://www.transfermarkt.com/transfermarkt/profil/spieler/'+raw_data.loc[:,'transferMarketID']
raw_data.transferMarketID.head()

0    https://www.transfermarkt.com/transfermarkt/profil/spieler/33793
1    https://www.transfermarkt.com/transfermarkt/profil/spieler/33793
2    https://www.transfermarkt.com/transfermarkt/profil/spieler/33793
3    https://www.transfermarkt.com/transfermarkt/profil/spieler/33793
4    https://www.transfermarkt.com/transfermarkt/profil/spieler/33793
Name: transferMarketID, dtype: object

In [194]:
len(raw_data.player.unique())

3289

### Helper Function

- Function to create a URI for the properties and if necessary also for the objects

In [195]:
def create_uri_ref(name:str, baseURL):
    """Takes a string and a baseURL then returns a URI extending the baseURL """
    
    quoted = quote(name.replace(" ","_"))
    return baseURL[quoted]
    

In [196]:
# takg.com -> time aware knowledge graph (takg) domain URL
baseURL = Namespace('http://takg.com/')
a = create_uri_ref('playedFor', baseURL)
a

rdflib.term.URIRef('http://takg.com/playedFor')

In [197]:
#pseudo code

##1. column mapping from dataset


##2. Build T-Box functions -->. the takg ontology will contain all the predicates


##3. Build A-box functions

In [198]:
raw_data.columns

Index(['player', 'fullName', 'team', 'teamCountry', 'DOB', 'sex',
       'transferMarketID', 'startTime', 'endTime'],
      dtype='object')

In [199]:
base_ont_col = ['player', 'fullName', 'transferMarketID', 'DOB', 'sex','team', 'teamCountry']

### Column Mappings

In [200]:
#ignore
for i in base_ont_col:
    print("'"+i+"'"+':create_uri_ref('+"'"+i+"'"',baseURL),')

'player':create_uri_ref('player',baseURL),
'fullName':create_uri_ref('fullName',baseURL),
'transferMarketID':create_uri_ref('transferMarketID',baseURL),
'DOB':create_uri_ref('DOB',baseURL),
'sex':create_uri_ref('sex',baseURL),
'team':create_uri_ref('team',baseURL),
'teamCountry':create_uri_ref('teamCountry',baseURL),


In [201]:
#the mapping will be used for creating teh predicate URI from column name
player_prop_URI_map = {
    
    'fullName':create_uri_ref('fullName',baseURL),
    'transferMarketID':create_uri_ref('transferMarketID',baseURL),
    'DOB':create_uri_ref('DOB',baseURL),
    'sex':create_uri_ref('sex',baseURL),
    'team':create_uri_ref('team',baseURL) 
}

team_prop_URI_map = {
    
    'teamCountry':create_uri_ref('teamCountry',baseURL)
    
}

In [202]:
#columns that have URIs unlike fullName & DOB whihc are Literal type
uri_features = ['transferMarketID','sex','team','teamCountry']
literal_features = ['fullName','DOB']

### Ontology/Schema

In [203]:
g=Graph() #graph object
g.bind("takg",baseURL) #Namespace

In [204]:
#T-Box for classes

#classes
g.add((baseURL.Player, RDF.type, RDFS.Class))
g.add((baseURL.Sex, RDF.type, RDFS.Class))
g.add((baseURL.Country, RDF.type, RDFS.Class))
g.add((baseURL.TFMID, RDF.type, RDFS.Class)) #tansfer market ID class
g.add((baseURL.FootballTeam, RDF.type, RDFS.Class))

#Subclass
g.add((baseURL.Player, RDFS.subClassOf, FOAF.Person))
g.add((baseURL.FootballTeam, RDFS.subClassOf, FOAF.Organization))

#OWL same as
g.add((baseURL.Player,OWL.sameAs,URIRef("https://www.wikidata.org/wiki/Q937857"))) #association football player
g.add((baseURL.Sex, OWL.sameAs,URIRef("https://www.wikidata.org/wiki/Q48264" ))) #gender Identity
g.add((baseURL.Country, OWL.sameAs, URIRef('https://www.wikidata.org/wiki/Q6256'))) #Country
g.add((baseURL.TFMID, OWL.sameAs, URIRef('https://www.wikidata.org/wiki/Q42253'))) #URL class
g.add((baseURL.FootballTeam, OWL.sameAs, URIRef('https://www.wikidata.org/wiki/Q476028'))) #Football Club
print(g.serialize(format='ttl'))

@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix takg: <http://takg.com/> .

takg:Country a rdfs:Class ;
    owl:sameAs <https://www.wikidata.org/wiki/Q6256> .

takg:FootballTeam a rdfs:Class ;
    rdfs:subClassOf <http://xmlns.com/foaf/0.1/Organization> ;
    owl:sameAs <https://www.wikidata.org/wiki/Q476028> .

takg:Player a rdfs:Class ;
    rdfs:subClassOf <http://xmlns.com/foaf/0.1/Person> ;
    owl:sameAs <https://www.wikidata.org/wiki/Q937857> .

takg:Sex a rdfs:Class ;
    owl:sameAs <https://www.wikidata.org/wiki/Q48264> .

takg:TFMID a rdfs:Class ;
    owl:sameAs <https://www.wikidata.org/wiki/Q42253> .




In [205]:
# funciton to add properties

def addProperty(g:Graph,class_uri:URIRef,class_prop_URI_map:dict,uri_cols:list,literal_cols:list):
    
    """The function takes in graph g, property and URI mapping dictionary, uri_obects dictionalry and 
    pleayer property to add object/ datatype properties.
    
    Arguments:
    
        g: graph object
        class_uri : The URI of the class to wich these properties belong
        class_prop_URI_map: dictionary with column name and it's URI mapping belonging to a specific class
        uri_cols : columns that have URI type values
        literal_cols : columns that have literal type values
        
    Output:
    
        Returns the graph object g by adding the properties
        
        """
    for prop, uri in class_prop_URI_map.items():
        #Object type properties
        if prop in uri_cols:
            g.add((uri, RDF.type, OWL.ObjectProperty))#property type
            g.add((uri,RDFS.domain,class_uri)) #domain
        #literal type (data type) properties    
        elif prop in literal_cols:
            g.add((uri, RDF.type, OWL.DatatypeProperty))#property type
            g.add((uri, RDFS.domain, class_uri)) #domain
    # return the graph        
    return g

In [206]:
# T- Box for properties
#adding the properties including their domains but not ranges
addProperty(g,baseURL.Player,player_prop_URI_map,uri_features,literal_features)
addProperty(g,baseURL.FootballTeam,team_prop_URI_map,uri_features,literal_features)
print(g.serialize(format='ttl'))

@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix takg: <http://takg.com/> .

takg:Country a rdfs:Class ;
    owl:sameAs <https://www.wikidata.org/wiki/Q6256> .

takg:FootballTeam a rdfs:Class ;
    rdfs:subClassOf <http://xmlns.com/foaf/0.1/Organization> ;
    owl:sameAs <https://www.wikidata.org/wiki/Q476028> .

takg:Player a rdfs:Class ;
    rdfs:subClassOf <http://xmlns.com/foaf/0.1/Person> ;
    owl:sameAs <https://www.wikidata.org/wiki/Q937857> .

takg:Sex a rdfs:Class ;
    owl:sameAs <https://www.wikidata.org/wiki/Q48264> .

takg:TFMID a rdfs:Class ;
    owl:sameAs <https://www.wikidata.org/wiki/Q42253> .

takg:DOB a owl:DatatypeProperty ;
    rdfs:domain takg:Player .

takg:fullName a owl:DatatypeProperty ;
    rdfs:domain takg:Player .

takg:sex a owl:ObjectProperty ;
    rdfs:domain takg:Player .

takg:team a owl:ObjectProperty ;
    rdfs:domain takg:Player .

takg:teamCountry a owl:ObjectProperty ;
    rdfs:domai

In [207]:
#T-Box for properties

#adding property ranges

g.add((create_uri_ref('fullName',baseURL), RDFS.range, XSD.string))
g.add((create_uri_ref('transferMarketID',baseURL), RDFS.range, baseURL.TFMID))
g.add((create_uri_ref('DOB',baseURL), RDFS.range, XSD.dateTimeStamp))
g.add((create_uri_ref('sex',baseURL), RDFS.range, baseURL.Sex))
g.add((create_uri_ref('team',baseURL), RDFS.range, baseURL.FootballTeam))
g.add((create_uri_ref('teamCountry',baseURL), RDFS.range, baseURL.Country))

#Define property subcalsses

#Define property owl:sameas here

<Graph identifier=N73cbaf69bd7e4cc0a95035ce426cdac1 (<class 'rdflib.graph.Graph'>)>

In [208]:
#A- Box

# define a functio to add the data from the database to the graph

def assertionBox(g:Graph,raw_data:pd.DataFrame,sub_col:str,class_prop_URI_map:dict):
    """
    The function adds triples to the graph object iteratively based on the input dataframe and the property mappings
    
    g:Graph: Graph object to which the triples will be added
    raw_data:pd.DataFrame : input pandas dataframe : The column names are property names except for the subject clumn
    sub:col : This is the name of the column in the dataframe whose values are considered to be subjects.
    class_prop_URI_map:dict : column name and its correspong URI mapping : The column names are predicates.
    
    """
    #reset the index of dataframe to avoid indexing errors
    raw_data = raw_data.reset_index()
    row = 0
    num_row = len(raw_data.index)
    #for each row in the dataframe
    while row<num_row:
        
        #get the record
        data_row = raw_data.loc[row,]
        #setting subject value
        subject = URIRef(data_row[sub_col])
        #loop for predicates and their corresponding values
        for label, predicate in class_prop_URI_map.items():
            #setting the object value
            obj_val   = data_row[label]
            # if object is a 
            if label in uri_features:
                g.add((subject,predicate,URIRef(obj_val)))
            #if object is a Literal
            elif label in literal_features:
                g.add((subject,predicate,Literal(obj_val)))
        row+=1


    return g


In [209]:
#Adding player and its linked data
assertionBox(g,raw_data,"player",player_prop_URI_map)

<Graph identifier=N73cbaf69bd7e4cc0a95035ce426cdac1 (<class 'rdflib.graph.Graph'>)>

In [210]:
#Adding Team and its country
assertionBox(g,raw_data,"team",team_prop_URI_map)

<Graph identifier=N73cbaf69bd7e4cc0a95035ce426cdac1 (<class 'rdflib.graph.Graph'>)>

In [211]:
col_class_dict = {
    
    'player':baseURL.Player,
    'transferMarketID':baseURL.TFMID,
    'sex':baseURL.Sex,
    'team':baseURL.FootballTeam,
    'teamCountry':baseURL.Country
    
}

In [212]:
col_class_dict['player']

rdflib.term.URIRef('http://takg.com/Player')

In [213]:
#linking all the objects to their classes.
# e.g <http://www.wikidata.org/entity/Q102027> a takg:Player
def entity_to_parent(g:Graph,raw_data:pd.DataFrame,col_class_dict:dict):
    """
    The function adds the (entity, RDF:type, parent) riples to the graph
    Parameters:
    
        g:Graph -> The graph object
        raw_data:pd.DataFrame -> database
        col_class_dict:dict -> dictionary to map whihc feature contains values belonging to whihc class
    
    Output:
        
        returns graph after adding the triples
    
    """
    #iterate through each key in the dictioanry
    for label, parent in col_class_dict.items():
        
        #pick corresponding column from the database and iterate over unique values and add triple
        for item in raw_data[label].unique():
            g.add((URIRef(item),RDF.type,parent))
    return g   

In [214]:
#calling the entity_to_parent function to add entity and its parent class relation
entity_to_parent(g,raw_data,col_class_dict)

<Graph identifier=N73cbaf69bd7e4cc0a95035ce426cdac1 (<class 'rdflib.graph.Graph'>)>

In [215]:
g.serialize(destination="./Data/takg.ttl", format="ttl", encoding="utf-8")
print(g.serialize(format='ttl'))

@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix takg: <http://takg.com/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

takg:Country a rdfs:Class ;
    owl:sameAs <https://www.wikidata.org/wiki/Q6256> .

takg:FootballTeam a rdfs:Class ;
    rdfs:subClassOf <http://xmlns.com/foaf/0.1/Organization> ;
    owl:sameAs <https://www.wikidata.org/wiki/Q476028> .

takg:Player a rdfs:Class ;
    rdfs:subClassOf <http://xmlns.com/foaf/0.1/Person> ;
    owl:sameAs <https://www.wikidata.org/wiki/Q937857> .

takg:Sex a rdfs:Class ;
    owl:sameAs <https://www.wikidata.org/wiki/Q48264> .

takg:TFMID a rdfs:Class ;
    owl:sameAs <https://www.wikidata.org/wiki/Q42253> .

takg:DOB a owl:DatatypeProperty ;
    rdfs:domain takg:Player ;
    rdfs:range xsd:dateTimeStamp .

takg:fullName a owl:DatatypeProperty ;
    rdfs:domain takg:Player ;
    rdfs:range xsd:string .

takg:sex a owl:ObjectProperty ;
    rdfs:domain takg:Player ;
    r